# SciGraphAgent Benchmark — Notebook 01
## Loading Benchmark Datasets: HotpotQA, MuSiQue, 2WikiMultiHopQA

---

**Notebook series:**
- Notebook 00 — Setup and foundations ✓ (done)
- **Notebook 01 — Loading benchmark datasets ← you are here**
- Notebook 02 — Building retrieval systems (next)
- Notebook 03 — Running experiments
- Notebook 04 — Metrics and results
- Notebook 05 — Visualisation

**What this notebook teaches:**
- What a benchmark dataset is and why we need one
- What multi-hop questions are and why they are hard
- What HuggingFace is and how the `datasets` library works
- What JSON is and how we store data
- How to explore and understand the data we downloaded
- How `step01_load_benchmarks.py` works — line by line

**Prerequisite:** Notebook 00 must be complete. Your virtual environment must be active and `GROQ_API_KEY` must be set in `.env`.

---

## Table of Contents

1. Environment setup
2. What is a benchmark dataset?
3. What is a multi-hop question?
4. The three datasets we use
5. What is HuggingFace?
6. What is the `datasets` library?
7. Understanding the data structure
8. What is JSON?
9. Running `step01_load_benchmarks.py` — line by line
10. Exploring the downloaded data
11. Why these datasets prove our novelty
12. Commit to GitHub

---
## Section 1 — Environment Setup

Run this cell first, every time you open this notebook. It loads the API key and verifies the environment is ready.

In [10]:
# ── Load environment variables from .env file ─────────────────────
# This must run before any other cell.
# Loads GROQ_API_KEY and HF_TOKEN into os.environ.

from dotenv import load_dotenv
import os
from pathlib import Path

# The notebook lives in notebooks/ but .env is in the project root (one level up)
# We search both locations to be safe
cwd = Path(os.getcwd())
print("Notebook running from:", cwd)

dotenv_path = cwd.parent / ".env"       # project root
if not dotenv_path.exists():
    dotenv_path = cwd / ".env"          # fallback: current dir

print("Loading .env from    :", dotenv_path)
print(".env found           :", dotenv_path.exists())

load_dotenv(dotenv_path)

# ── Verify keys loaded ────────────────────────────────────────────
print()
groq_key = os.environ.get("GROQ_API_KEY", "")
hf_token  = os.environ.get("HF_TOKEN", "")

print("GROQ_API_KEY:", groq_key[:8] + "****" if groq_key else "NOT FOUND")
print("HF_TOKEN    :", hf_token[:8] + "****" if hf_token else "NOT FOUND")

# ── Verify working directory and data folder ──────────────────────
print()
data_dir = cwd.parent / "data" if (cwd.parent / "data").exists() else cwd / "data"
if data_dir.exists():
    files = list(data_dir.glob("*.json"))
    print(f"✓ data/ folder found: {data_dir}")
    print(f"  {len(files)} JSON files inside:")
    for f in sorted(files):
        size_kb = f.stat().st_size // 1024
        print(f"    {f.name:<45} {size_kb} KB")
else:
    print("  data/ folder not found — will be created when step01 runs")

print("\n✓ Environment ready")

Notebook running from: /run/media/bala/HDD/Projects/scigraphagent-benchmark
Loading .env from    : /run/media/bala/HDD/Projects/scigraphagent-benchmark/.env
.env found           : True

GROQ_API_KEY: gsk_GWHq****
HF_TOKEN    : hf_kkSXQ****

✓ data/ folder found: /run/media/bala/HDD/Projects/scigraphagent-benchmark/data
  4 JSON files inside:
    2wikimultihopqa_sample_3.json                 9 KB
    hotpotqa_sample_3.json                        19 KB
    manifest.json                                 0 KB
    musique_sample_3.json                         6 KB

✓ Environment ready


---
## Section 2 — What is a Benchmark Dataset?

### The problem with measuring AI quality

How do you know if one AI system is better than another? You could ask both systems a question and judge the answers yourself — but that is slow, expensive, and different people might disagree on what a "good" answer is.

Researchers solved this with **benchmark datasets**.

### What is a benchmark dataset?

A benchmark dataset is a large collection of questions where:
- Every question has a **known correct answer** (written by humans)
- The questions are **standardised** — the same questions are used by everyone
- The **scoring is automatic** — a computer compares the AI's answer to the correct answer

Because every researcher uses the same questions and the same scoring method, you can directly compare results across different papers. When a paper says "our system scores 0.62 F1 on HotpotQA", you can immediately compare that to any other paper that evaluated on HotpotQA.

### Why we use three benchmarks

Using multiple benchmarks makes the results more trustworthy. If a system performs well on all three, it is not just getting lucky on one particular type of question.

### What is F1 score?

F1 is a number between 0.0 and 1.0 that measures how much the AI's answer overlaps with the correct answer in terms of words.

- Gold answer: `"Albert Einstein"`
- AI answer A: `"Einstein"` → partial overlap → F1 = 0.67
- AI answer B: `"Albert Einstein was born in Germany"` → partial overlap → F1 = 0.50  
- AI answer C: `"Albert Einstein"` → perfect match → F1 = 1.0
- AI answer D: `"Isaac Newton"` → no overlap → F1 = 0.0

In [2]:
# Let's implement F1 score from scratch to understand how it works

def normalise(text):
    """
    Clean text for fair comparison:
    - lowercase everything
    - remove articles (a, an, the)
    - remove punctuation
    - collapse extra spaces
    This is the standard normalisation used by HotpotQA evaluation.
    """
    import re
    text = text.lower().strip()
    text = re.sub(r'\b(a|an|the)\b', ' ', text)   # remove articles
    text = re.sub(r'[^a-z0-9\s]', '', text)        # keep only letters, numbers, spaces
    return re.sub(r'\s+', ' ', text).strip()        # collapse spaces


def f1_score(prediction, gold):
    """
    Token-overlap F1 score.
    Precision = fraction of predicted words that are correct.
    Recall    = fraction of gold words that appear in prediction.
    F1        = harmonic mean of precision and recall.
    """
    pred_tokens = normalise(prediction).split()
    gold_tokens = normalise(gold).split()

    if not pred_tokens or not gold_tokens:
        return 0.0

    # Count words that appear in both (intersection)
    common = set(pred_tokens) & set(gold_tokens)
    if not common:
        return 0.0

    precision = len(common) / len(pred_tokens)   # how many predicted words are correct?
    recall    = len(common) / len(gold_tokens)   # how many gold words did we find?
    f1        = 2 * precision * recall / (precision + recall)
    return round(f1, 3)


# Test it
gold = "Albert Einstein"
predictions = [
    "Albert Einstein",
    "Einstein",
    "Albert Einstein was born in Germany",
    "Isaac Newton",
    "I don't know",
]

print(f"Gold answer: '{gold}'\n")
print(f"{'Prediction':<40} {'F1':>6}")
print("-" * 48)
for pred in predictions:
    score = f1_score(pred, gold)
    bar = "█" * int(score * 20)
    print(f"'{pred}'{'':>{38-len(pred)}} {score:>6.3f}  {bar}")

Gold answer: 'Albert Einstein'

Prediction                                   F1
------------------------------------------------
'Albert Einstein'                         1.000  ████████████████████
'Einstein'                                0.667  █████████████
'Albert Einstein was born in Germany'     0.500  ██████████
'Isaac Newton'                            0.000  
'I don't know'                            0.000  


---
## Section 3 — What is a Multi-Hop Question?

### Single-hop vs multi-hop

A **single-hop** question can be answered from one piece of text:
> "What is the capital of France?" → Answer: "Paris" (in one sentence anywhere)

A **multi-hop** question requires connecting information from two or more separate texts:
> "What is the capital of the country where the Eiffel Tower is located?"

To answer this, you need two facts:
1. The Eiffel Tower is in France (Fact A)
2. The capital of France is Paris (Fact B)

No single passage contains both facts together. You must **hop** from one to the other.

### Why standard RAG fails on multi-hop questions

Standard RAG finds passages that are **similar** to the question. When it searches for "capital of country where Eiffel Tower is located", it might find:
- "The Eiffel Tower is an iron lattice tower in Paris, France" → contains the answer implicitly

But for harder questions, no single passage contains all the hops. That is when standard RAG fails and graph-based retrieval wins — the graph explicitly connects the entities across passages.

### The three hop types in our datasets

In [5]:
import json
from pathlib import Path

data_dir = Path("data")
examples = {}

for dataset in ["hotpotqa", "musique", "2wikimultihopqa"]:
    sample_file = data_dir / f"{dataset}_sample_3.json"
    if sample_file.exists():                          # ← indented INSIDE the for loop
        with open(sample_file) as f:
            records = json.load(f)
        examples[dataset] = records[0]

if examples:
    print("=" * 65)
    print("EXAMPLES FROM OUR DOWNLOADED DATA")
    print("=" * 65)

    for dataset, rec in examples.items():
        print(f"\n[{dataset.upper()}]")
        print(f"  Question type : {rec.get('q_type', 'unknown')}")
        print(f"  Question      : {rec['question']}")
        print(f"  Gold answer   : {rec['answer']}")

        # Defensive string conversion — context must be a string
        ctx = rec['context']
        if not isinstance(ctx, str):
            ctx = "\n".join(str(p) for p in ctx) if isinstance(ctx, list) else str(ctx)

        print(f"  Context length: {len(ctx)} characters")
        preview = ctx[:300].replace("\n", " | ")
        print(f"  Context preview:")
        print(f"    {preview}...")
else:
    print("No data files found yet.")
    print("Run Section 9 first to download the datasets.")

EXAMPLES FROM OUR DOWNLOADED DATA

[HOTPOTQA]
  Question type : bridge
  Question      : What nationality was Oliver Reed's character in the film Royal Flash?
  Gold answer   : Prussian
  Context length: 5023 characters
  Context preview:
    [Robin Barton] Robin Barton (born 5 November 1958) is a British art dealer dealing primarily with Banksy's.  Barton studied photography and graphic design at the Exeter College of Art and Design and this was his first encounter with Russell Young.  Moving to London in 1980 he began working as a free...

[MUSIQUE]
  Question type : 3-hop
  Question      : Who is the president of the newly declared independent country that is part of the Commission of Truth and Friendship with the country where Kotamadya is located?
  Gold answer   : Francisco Guterres
  Context length: 3506 characters
  Context preview:
    Zeferino Martins, also known as Ze Martins (born September 5, 1985) is an East Timorese footballer who plays as midfielder for Ad. Dili Oeste a

---
## Section 4 — The Three Datasets We Use

### Why these three specifically?

These are the **standard benchmarks** for multi-hop QA evaluation. Every major Graph-RAG paper uses them:
- Microsoft GraphRAG (Edge et al., 2024)
- LightRAG (Guo et al., 2025)
- HippoRAG (Gutiérrez et al., 2024)
- Graph-R1 (Luo et al., 2026)
- GraphRAG-R1 (Yu et al., 2026)

By using the same benchmarks, our results can be directly compared to published numbers.

### HotpotQA — the classic

**Paper:** Yang et al., EMNLP 2018, arXiv:1809.09600  
**Size:** 113,000 question-answer pairs  
**Source:** Wikipedia  
**Type:** Two question types:
- **Bridge:** Entity A relates to Entity B which gives the answer (2 hops)
- **Comparison:** Compare two entities on the same property (2 hops)

Example bridge question: *"What nationality was the director of the film that stars both X and Y?"*  
Hop 1: Find the film → Hop 2: Find the director's nationality

### MuSiQue — the hardest

**Paper:** Trivedi et al., TACL 2022, arXiv:2108.00573  
**Size:** 25,000 questions  
**Source:** Wikipedia  
**Type:** 2, 3, or 4 hops — compositional reasoning chains

MuSiQue was specifically designed to be harder than HotpotQA. The questions cannot be answered by finding one relevant passage — every hop is required. A single-hop model scores 30 F1 points lower on MuSiQue than a proper multi-hop model.

### 2WikiMultiHopQA — the diverse

**Paper:** Ho et al., COLING 2020, arXiv:2011.01060  
**Size:** 192,000 questions  
**Source:** Wikipedia + Wikidata  
**Types:** bridge, comparison, compositional, inference

The largest of the three. Uses structured Wikidata facts alongside Wikipedia text, making it more diverse in question types.

In [6]:
# Compare the three datasets side by side

dataset_info = [
    {
        "name":       "HotpotQA",
        "paper":      "Yang et al., EMNLP 2018",
        "arxiv":      "1809.09600",
        "total_size": 113_000,
        "hf_id":      "hotpotqa/hotpot_qa",
        "hops":       "2",
        "q_types":    "bridge, comparison",
        "difficulty": "medium",
    },
    {
        "name":       "MuSiQue",
        "paper":      "Trivedi et al., TACL 2022",
        "arxiv":      "2108.00573",
        "total_size": 25_000,
        "hf_id":      "bdsaglam/musique",
        "hops":       "2-4",
        "q_types":    "compositional",
        "difficulty": "hardest",
    },
    {
        "name":       "2WikiMultiHopQA",
        "paper":      "Ho et al., COLING 2020",
        "arxiv":      "2011.01060",
        "total_size": 192_000,
        "hf_id":      "framolfese/2WikiMultihopQA",
        "hops":       "2+",
        "q_types":    "bridge, comparison, compositional, inference",
        "difficulty": "medium-hard",
    },
]

print(f"{'Dataset':<20} {'Total Qs':>10} {'Hops':>6} {'Difficulty':<12} HuggingFace ID")
print("-" * 80)
for d in dataset_info:
    print(f"{d['name']:<20} {d['total_size']:>10,} {d['hops']:>6} {d['difficulty']:<12} {d['hf_id']}")

print()
total = sum(d['total_size'] for d in dataset_info)
print(f"Total questions available: {total:,}")
print(f"We sample              : 50 per dataset (3 for smoke test)")
print(f"Paper protocol uses    : 1,000 per dataset")

Dataset                Total Qs   Hops Difficulty   HuggingFace ID
--------------------------------------------------------------------------------
HotpotQA                113,000      2 medium       hotpotqa/hotpot_qa
MuSiQue                  25,000    2-4 hardest      bdsaglam/musique
2WikiMultiHopQA         192,000     2+ medium-hard  framolfese/2WikiMultihopQA

Total questions available: 330,000
We sample              : 50 per dataset (3 for smoke test)
Paper protocol uses    : 1,000 per dataset


---
## Section 5 — What is HuggingFace?

### The analogy

Think of HuggingFace as **GitHub, but for AI**.

GitHub stores code that anyone can download and use. HuggingFace stores:
- **Models** — pre-trained AI models (like the embedding model we use)
- **Datasets** — collections of data for training and evaluating AI (like our benchmarks)
- **Spaces** — live demos of AI applications

HuggingFace is at `huggingface.co`. Our three datasets are hosted there.

### How data is stored on HuggingFace

Datasets on HuggingFace are stored in **Parquet** format. Parquet is a compressed, columnar file format that stores tabular data efficiently. Think of it like a very efficient spreadsheet.

When you call `load_dataset()`, the `datasets` library:
1. Downloads the Parquet file from HuggingFace
2. Caches it locally in `~/.cache/huggingface/` (so it does not re-download)
3. Gives you a Python object you can query like a database

### The HuggingFace token warning

You saw this message when running step01:
```
Warning: You are sending unauthenticated requests to the HF Hub.
Please set a HF_TOKEN to enable higher rate limits and faster downloads.
```

This is a warning, not an error. All three datasets are public and work without a token. With a free HuggingFace account and token, downloads are faster. For our project, the warning can be safely ignored — the downloads work fine.

In [7]:
# Check the HuggingFace cache — where downloaded datasets are stored

import os
from pathlib import Path

# HuggingFace stores downloaded data here by default
hf_cache = Path.home() / ".cache" / "huggingface" / "datasets"

print("HuggingFace dataset cache location:")
print(" ", hf_cache)
print()

if hf_cache.exists():
    # Calculate total cache size
    total_bytes = sum(f.stat().st_size for f in hf_cache.rglob("*") if f.is_file())
    total_mb = total_bytes / (1024 * 1024)
    
    # List cached datasets
    cached = [d.name for d in hf_cache.iterdir() if d.is_dir()]
    
    print(f"Total cache size: {total_mb:.0f} MB")
    print(f"Cached datasets ({len(cached)}):")
    for name in sorted(cached):
        print(f"  - {name}")
    print()
    print("These files are already downloaded.")
    print("Running step01 again will use the cache (no re-download).")
else:
    print("Cache is empty — datasets will be downloaded when step01 runs.")
    print(f"Approximate download sizes:")
    print(f"  HotpotQA       : ~360 MB")
    print(f"  MuSiQue        : ~540 MB")
    print(f"  2WikiMultiHopQA: ~390 MB")
    print(f"  Total          : ~1.3 GB")

HuggingFace dataset cache location:
  /home/bala/.cache/huggingface/datasets

Total cache size: 1757 MB
Cached datasets (3):
  - bdsaglam___musique
  - framolfese___2_wiki_multihop_qa
  - hotpotqa___hotpot_qa

These files are already downloaded.
Running step01 again will use the cache (no re-download).


---
## Section 6 — What is the `datasets` Library?

The `datasets` library is made by HuggingFace. It is the standard tool for loading and working with AI benchmark datasets in Python.

### The main function: `load_dataset()`

```python
from datasets import load_dataset

ds = load_dataset(
    "hotpotqa/hotpot_qa",   # dataset ID on HuggingFace
    "distractor",           # configuration (some datasets have variants)
    split="validation"      # which split to load (train / validation / test)
)
```

### What is a split?

Most ML datasets are divided into three parts:

| Split | Purpose | Size (HotpotQA) |
|---|---|---|
| `train` | Used to train models | 90,447 questions |
| `validation` | Used to evaluate during development | 7,405 questions |
| `test` | Used for final evaluation | Not public |

We always use `validation` — it is the standard split for reporting results in papers, and the test split answers are not publicly available.

### What is the `distractor` configuration?

HotpotQA has two versions:
- `distractor`: 10 context passages are provided (2 relevant + 8 random distractors). Harder.
- `fullwiki`: The full Wikipedia is the context. Hardest.

We use `distractor` — it is the standard evaluation setup used in all published papers.

In [11]:
# Demonstrate the datasets library on a tiny sample
# (uses cache if already downloaded — no new download needed)

from datasets import load_dataset

print("Loading HotpotQA validation split (uses cache)...")
ds = load_dataset(
    "hotpotqa/hotpot_qa",
    "distractor",
    split="validation"
)

print()
print("Dataset object type:", type(ds))
print("Number of examples :", len(ds))
print("Column names       :", ds.column_names)
print()

# Access the first example like a dictionary
first = ds[0]
print("First example keys :", list(first.keys()))
print()
print("Question:", first["question"])
print("Answer  :", first["answer"])
print("Type    :", first["type"])
print("ID      :", first["id"])

Loading HotpotQA validation split (uses cache)...

Dataset object type: <class 'datasets.arrow_dataset.Dataset'>
Number of examples : 7405
Column names       : ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context']

First example keys : ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context']

Question: Were Scott Derrickson and Ed Wood of the same nationality?
Answer  : yes
Type    : comparison
ID      : 5a8b57f25542995d1e6f1371


In [12]:
# Demonstrate shuffle and select — the two operations in step01

print("=" * 55)
print("SHUFFLE AND SELECT — how we sample n questions")
print("=" * 55)

# Without shuffle — first 3 questions would always be the same
first_3_unshuffled = [ds[i]["question"][:60] for i in range(3)]
print("\nFirst 3 WITHOUT shuffle (always the same):")
for i, q in enumerate(first_3_unshuffled):
    print(f"  {i+1}. {q}...")

# With shuffle + seed=42 — reproducible random sample
# seed=42 means: the random shuffle is always the same.
# This is critical for reproducibility — re-running the script
# gives exactly the same sample every time.
shuffled = ds.shuffle(seed=42)
sample_3 = shuffled.select(range(3))   # take first 3 from shuffled

print("\nFirst 3 WITH shuffle(seed=42) — same every run:")
for i, row in enumerate(sample_3):
    print(f"  {i+1}. {row['question'][:60]}...")

print()
print("Why seed=42?")
print("  Any fixed number works. 42 is a tradition in ML research.")
print("  The important thing is that it is fixed — not that it is 42.")
print("  With the same seed, you get the same shuffle every time you run.")

SHUFFLE AND SELECT — how we sample n questions

First 3 WITHOUT shuffle (always the same):
  1. Were Scott Derrickson and Ed Wood of the same nationality?...
  2. What government position was held by the woman who portrayed...
  3. What science fantasy young adult series, told in first perso...

First 3 WITH shuffle(seed=42) — same every run:
  1. What nationality was Oliver Reed's character in the film Roy...
  2. Pacific Mozart Ensemble performed which German composer's De...
  3. Who released the song "With or Without You" first, Jai McDow...

Why seed=42?
  Any fixed number works. 42 is a tradition in ML research.
  The important thing is that it is fixed — not that it is 42.
  With the same seed, you get the same shuffle every time you run.


---
## Section 7 — Understanding the Data Structure

Each dataset has a different internal structure. Our loader functions extract the fields we need into a consistent format.

### The target format — what our loader produces

Every record we save has exactly these 6 fields:

```python
{
    "id":       "unique question identifier",
    "question": "the multi-hop question text",
    "answer":   "the gold correct answer",
    "context":  "the text passages containing evidence",
    "dataset":  "hotpotqa" / "musique" / "2wikimultihopqa",
    "q_type":   "bridge" / "comparison" / "2-hop" / etc.
}
```

This consistent format means all downstream steps (steps 02-06) can work the same way regardless of which dataset the question came from.

### HotpotQA's context structure — the tricky part

HotpotQA stores context as a nested structure:

```python
context = {
    "title":     ["Article 1 title", "Article 2 title", ...],
    "sentences": [["Sent 1a", "Sent 1b"], ["Sent 2a", "Sent 2b"], ...]
}
```

Each entry in `title` corresponds to an entry in `sentences`. Our loader flattens this into plain text.

In [13]:
# Show the raw structure of each dataset and what our loader does with it

print("=" * 60)
print("RAW HOTPOTQA CONTEXT STRUCTURE")
print("=" * 60)

raw_example = ds[0]  # first example from HotpotQA

print("\nRaw context field type:", type(raw_example["context"]))
print("Context keys:", list(raw_example["context"].keys()))
print()

# Show the raw nested structure
titles    = raw_example["context"]["title"]
sentences = raw_example["context"]["sentences"]
print(f"Number of passages: {len(titles)}")
print()
for i, (title, sents) in enumerate(zip(titles[:3], sentences[:3])):
    print(f"Passage {i+1}: [{title}]")
    for j, sent in enumerate(sents[:2]):
        print(f"  Sentence {j+1}: {sent[:80]}...")
    print()

print("=" * 60)
print("AFTER OUR LOADER PROCESSES IT")
print("=" * 60)

# This is exactly what load_hotpotqa() does
passages = []
for title, sents in zip(titles, sentences):
    flat_text = f"[{title}] " + " ".join(sents)
    passages.append(flat_text)

context = "\n".join(passages[:10])  # cap at 10 passages
print()
print("Flat context (first 500 chars):")
print(context[:500])
print("...")
print(f"\nTotal context length: {len(context)} characters")

RAW HOTPOTQA CONTEXT STRUCTURE

Raw context field type: <class 'dict'>
Context keys: ['title', 'sentences']

Number of passages: 10

Passage 1: [Ed Wood (film)]
  Sentence 1: Ed Wood is a 1994 American biographical period comedy-drama film directed and pr...
  Sentence 2:  The film concerns the period in Wood's life when he made his best-known films a...

Passage 2: [Scott Derrickson]
  Sentence 1: Scott Derrickson (born July 16, 1966) is an American director, screenwriter and ...
  Sentence 2:  He lives in Los Angeles, California....

Passage 3: [Woodson, Arkansas]
  Sentence 1: Woodson is a census-designated place (CDP) in Pulaski County, Arkansas, in the U...
  Sentence 2:  Its population was 403 at the 2010 census....

AFTER OUR LOADER PROCESSES IT

Flat context (first 500 chars):
[Ed Wood (film)] Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood.  The film concerns the period in 

---
## Section 8 — What is JSON?

### Why we use JSON

After downloading and processing the datasets, we save them as JSON files. All downstream steps read from these files.

**JSON (JavaScript Object Notation)** is a simple text format for storing structured data. It looks like Python dictionaries and lists.

### JSON syntax

```json
{
  "name": "SciGraphAgent",
  "version": 1,
  "active": true,
  "datasets": ["hotpotqa", "musique", "2wikimultihopqa"],
  "config": {
    "alpha": 0.6,
    "max_retries": 2
  }
}
```

Rules:
- Strings must use **double quotes** (not single quotes)
- `true` / `false` (lowercase, not Python's `True` / `False`)
- No trailing commas after the last item

### Python's `json` library

Python converts between JSON (text) and Python objects automatically:
- `json.dump(data, file)` → write Python object to JSON file
- `json.load(file)` → read JSON file into Python object
- `json.dumps(data)` → convert Python object to JSON string
- `json.loads(string)` → convert JSON string to Python object

In [14]:
import json
from pathlib import Path

# Demonstrate JSON read/write
print("=" * 55)
print("JSON DEMONSTRATION")
print("=" * 55)

# A Python dictionary
sample_record = {
    "id":       "demo_001",
    "question": "Who directed the film featuring the actor born in Prussian Germany?",
    "answer":   "Richard Lester",
    "dataset":  "hotpotqa",
    "q_type":   "bridge"
}

# Convert to JSON string
json_string = json.dumps(sample_record, indent=2)
print("Python dict → JSON string:")
print(json_string)

# Convert back to Python
recovered = json.loads(json_string)
print("\nJSON string → Python dict:")
print(f"  question: {recovered['question']}")
print(f"  answer  : {recovered['answer']}")
print(f"  Types recovered correctly: {type(recovered) == dict}")

# Read an actual saved file
print()
print("=" * 55)
print("READING OUR SAVED FILES")
print("=" * 55)

for dataset in ["hotpotqa", "musique", "2wikimultihopqa"]:
    path = Path("data") / f"{dataset}_sample_3.json"
    if path.exists():
        with open(path) as f:
            records = json.load(f)
        print(f"\n{dataset}_sample_3.json:")
        print(f"  Type    : {type(records)}")
        print(f"  Length  : {len(records)} records")
        print(f"  Keys    : {list(records[0].keys())}")
        print(f"  Answer  : {records[0]['answer']}")
    else:
        print(f"\n{dataset}: file not found — run Section 9 first")

JSON DEMONSTRATION
Python dict → JSON string:
{
  "id": "demo_001",
  "question": "Who directed the film featuring the actor born in Prussian Germany?",
  "answer": "Richard Lester",
  "dataset": "hotpotqa",
  "q_type": "bridge"
}

JSON string → Python dict:
  question: Who directed the film featuring the actor born in Prussian Germany?
  answer  : Richard Lester
  Types recovered correctly: True

READING OUR SAVED FILES

hotpotqa_sample_3.json:
  Type    : <class 'list'>
  Length  : 3 records
  Keys    : ['id', 'question', 'answer', 'context', 'dataset', 'q_type']
  Answer  : Prussian

musique_sample_3.json:
  Type    : <class 'list'>
  Length  : 3 records
  Keys    : ['id', 'question', 'answer', 'context', 'dataset', 'q_type']
  Answer  : Francisco Guterres

2wikimultihopqa_sample_3.json:
  Type    : <class 'list'>
  Length  : 3 records
  Keys    : ['id', 'question', 'answer', 'context', 'dataset', 'q_type']
  Answer  : Taiyuan


---
## Section 9 — Running `step01_load_benchmarks.py` — Line by Line

Now we run the actual script that downloads all three datasets. We also walk through every part of the code to understand exactly what it does.

### The script structure

```
step01_load_benchmarks.py
│
├── Constants (DATA_DIR, RANDOM_SEED)
├── load_hotpotqa(n)       ← downloads + processes HotpotQA
├── load_musique(n)        ← downloads + processes MuSiQue
├── load_2wikimultihopqa(n) ← downloads + processes 2WikiMultiHopQA
├── print_summary(records, name) ← prints stats about the sample
└── main(n)               ← orchestrates all three loaders
    └── if __name__ == "__main__": ← reads --n argument from command line
```

### What `if __name__ == "__main__":` means

This is a Python convention. When Python runs a file directly (`python step01.py`), it sets `__name__` to `"__main__"`. When another file imports it (`import step01`), `__name__` is set to `"step01"` instead.

This means: "only run `main()` when this file is run directly — not when it is imported by another script."

### What `argparse` does

`argparse` reads command-line arguments. When you run:
```bash
python step01_load_benchmarks.py --n 3
```
`argparse` reads `--n 3`, converts `"3"` to the integer `3`, and makes it available as `args.n`.

In [15]:
# Run step01 directly from the notebook with n=3
# This mirrors running: python step01_load_benchmarks.py --n 3
#
# We import the functions from the script and call main() directly.
# This is the same code the terminal runs — no duplication.

import sys
sys.path.insert(0, "..")  # add project root to Python path

# Import and run
import importlib.util
from pathlib import Path

# Load the script as a module
script_path = Path(".").parent / "step01_load_benchmarks.py"
if not script_path.exists():
    script_path = Path("step01_load_benchmarks.py")

print(f"Loading script from: {script_path.resolve()}")
print()

spec = importlib.util.spec_from_file_location("step01", script_path)
step01 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(step01)

# Run with n=3 (smoke test size)
print("Running step01.main(n=3)...")
print()
step01.main(n=3)

Loading script from: /run/media/bala/HDD/Projects/scigraphagent-benchmark/step01_load_benchmarks.py

Running step01.main(n=3)...


  STEP 01: LOAD BENCHMARK DATASETS
  Sample size : 3 questions per dataset
  Output dir  : /run/media/bala/HDD/Projects/scigraphagent-benchmark/data
  Random seed : 42 (reproducible)


[HOTPOTQA]

  [hotpotqa] 3 questions sampled
    bridge                        : 3

  Example question:
    Q: What nationality was Oliver Reed's character in the film Royal Flash?
    A: Prussian

  Saved → data/hotpotqa_sample_3.json

[MUSIQUE]

  [musique] 3 questions sampled
    3-hop                         : 2
    2-hop                         : 1

  Example question:
    Q: Who is the president of the newly declared independent country that is part of the Commission of Truth and Friendship with the country where Kotamadya is located?
    A: Francisco Guterres

  Saved → data/musique_sample_3.json

[2WIKIMULTIHOPQA]

  [2wikimultihopqa] 3 questions sampled
    compositi

### Code walkthrough — the key parts explained

In [16]:
# ── PART 1: Constants ─────────────────────────────────────────────
# These lines appear at the top of step01_load_benchmarks.py

from pathlib import Path

DATA_DIR    = Path("data")
DATA_DIR.mkdir(exist_ok=True)
RANDOM_SEED = 42

# Path("data") creates a Path object pointing to the data/ folder
# .mkdir(exist_ok=True) creates the folder if it doesn't exist
#   - exist_ok=True means: don't raise an error if it already exists
#   - Without this, running step01 twice would crash on the second run

print("DATA_DIR:", DATA_DIR)
print("DATA_DIR absolute path:", DATA_DIR.resolve())
print("DATA_DIR exists:", DATA_DIR.exists())
print("RANDOM_SEED:", RANDOM_SEED)

# Constructing file paths
example_path = DATA_DIR / "hotpotqa_sample_3.json"
print()
print("Path construction:")
print("  DATA_DIR / 'hotpotqa_sample_3.json' =", example_path)
print("  The / operator joins path components — works on all OS")

DATA_DIR: data
DATA_DIR absolute path: /run/media/bala/HDD/Projects/scigraphagent-benchmark/data
DATA_DIR exists: True
RANDOM_SEED: 42

Path construction:
  DATA_DIR / 'hotpotqa_sample_3.json' = data/hotpotqa_sample_3.json
  The / operator joins path components — works on all OS


In [17]:
# ── PART 2: What load_hotpotqa() does step by step ────────────────

from datasets import load_dataset

print("Step 1: load_dataset() downloads + loads from cache")
ds = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation")
print(f"  Loaded {len(ds)} validation examples")

print()
print("Step 2: shuffle(seed=42) randomises the order")
shuffled = ds.shuffle(seed=42)
print(f"  First question before shuffle: {ds[0]['question'][:50]}...")
print(f"  First question after  shuffle: {shuffled[0]['question'][:50]}...")

print()
print("Step 3: select(range(3)) takes the first 3")
sample = shuffled.select(range(3))
print(f"  Sample size: {len(sample)}")

print()
print("Step 4: extract fields into our standard format")
for i, row in enumerate(sample):
    # Build flat context from nested structure
    passages = []
    for title, sents in zip(row["context"]["title"], row["context"]["sentences"]):
        passages.append(f"[{title}] " + " ".join(sents))
    context = "\n".join(passages[:10])
    
    record = {
        "id":       row["id"],
        "question": row["question"],
        "answer":   row["answer"],
        "context":  context,
        "dataset":  "hotpotqa",
        "q_type":   row.get("type", "unknown"),
    }
    print(f"  Record {i+1}: Q='{record['question'][:45]}...' A='{record['answer']}'")

Step 1: load_dataset() downloads + loads from cache
  Loaded 7405 validation examples

Step 2: shuffle(seed=42) randomises the order
  First question before shuffle: Were Scott Derrickson and Ed Wood of the same nati...
  First question after  shuffle: What nationality was Oliver Reed's character in th...

Step 3: select(range(3)) takes the first 3
  Sample size: 3

Step 4: extract fields into our standard format
  Record 1: Q='What nationality was Oliver Reed's character ...' A='Prussian'
  Record 2: Q='Pacific Mozart Ensemble performed which Germa...' A='Kurt Julian Weill'
  Record 3: Q='Who released the song "With or Without You" f...' A='U2'


In [18]:
# ── PART 3: The manifest file ────────────────────────────────────
# step01 saves a manifest.json that tells downstream steps
# where to find the data files.

import json
from pathlib import Path

manifest_path = Path("data") / "manifest.json"

if manifest_path.exists():
    with open(manifest_path) as f:
        manifest = json.load(f)
    
    print("manifest.json contents:")
    print(json.dumps(manifest, indent=2))
    print()
    print("How downstream steps use this:")
    print("  with open('data/manifest.json') as f:")
    print("      manifest = json.load(f)")
    print("  n = manifest['sample_n']        # know how many questions")
    print("  paths = manifest['datasets']    # find each file")
else:
    print("manifest.json not found — run Section 9 (run step01) first")

manifest.json contents:
{
  "created": "2026-08-31T15:52:18.706222",
  "sample_n": 3,
  "seed": 42,
  "datasets": {
    "hotpotqa": "data/hotpotqa_sample_3.json",
    "musique": "data/musique_sample_3.json",
    "2wikimultihopqa": "data/2wikimultihopqa_sample_3.json"
  }
}

How downstream steps use this:
  with open('data/manifest.json') as f:
      manifest = json.load(f)
  n = manifest['sample_n']        # know how many questions
  paths = manifest['datasets']    # find each file


---
## Section 10 — Exploring the Downloaded Data

Now that the data is saved, let's explore it properly. Understanding the data is critical before building any system that processes it.

In [19]:
import json
from pathlib import Path
from collections import Counter

data_dir = Path("data")

all_records = []
dataset_counts = {}

for dataset in ["hotpotqa", "musique", "2wikimultihopqa"]:
    path = data_dir / f"{dataset}_sample_3.json"
    if path.exists():
        with open(path) as f:
            records = json.load(f)
        all_records.extend(records)
        dataset_counts[dataset] = len(records)

print("=" * 60)
print("DATA OVERVIEW")
print("=" * 60)
print(f"\nTotal records loaded: {len(all_records)}")
for ds_name, count in dataset_counts.items():
    print(f"  {ds_name:<25}: {count} questions")

# Question length analysis
q_lengths = [len(r["question"].split()) for r in all_records]
a_lengths = [len(r["answer"].split())   for r in all_records]
c_lengths = [len(r["context"].split())  for r in all_records]

print()
print("=" * 60)
print("LENGTH STATISTICS (in words)")
print("=" * 60)
print(f"\n{'Metric':<20} {'Min':>6} {'Max':>6} {'Avg':>8}")
print("-" * 44)
for name, lengths in [("Question", q_lengths), ("Answer", a_lengths), ("Context", c_lengths)]:
    print(f"{name:<20} {min(lengths):>6} {max(lengths):>6} {sum(lengths)/len(lengths):>8.1f}")

# Question type distribution
print()
print("=" * 60)
print("QUESTION TYPE DISTRIBUTION")
print("=" * 60)
type_counts = Counter(r["q_type"] for r in all_records)
for qtype, count in type_counts.most_common():
    bar = "█" * count
    print(f"  {qtype:<20} {count:>3}  {bar}")

DATA OVERVIEW

Total records loaded: 9
  hotpotqa                 : 3 questions
  musique                  : 3 questions
  2wikimultihopqa          : 3 questions

LENGTH STATISTICS (in words)

Metric                  Min    Max      Avg
--------------------------------------------
Question                  9     27     14.8
Answer                    1      7      2.1
Context                  63   1408    608.9

QUESTION TYPE DISTRIBUTION
  bridge                 3  ███
  3-hop                  2  ██
  compositional          2  ██
  2-hop                  1  █
  comparison             1  █


In [20]:
# Display all questions with their answers nicely

print("=" * 65)
print("ALL QUESTIONS IN OUR SAMPLE")
print("=" * 65)

for i, rec in enumerate(all_records):
    print(f"\n[{i+1}] [{rec['dataset'].upper()}] [{rec['q_type']}]")
    print(f"  Q: {rec['question']}")
    print(f"  A: {rec['answer']}")
    
    # Show the context that contains the answer
    # (check if the answer appears anywhere in the context)
    answer_in_context = rec['answer'].lower() in rec['context'].lower()
    print(f"  Answer in context: {answer_in_context}")
    
    # How many hops does this question need?
    q_type = rec['q_type']
    if 'hop' in q_type:
        hops = q_type.split('-')[0]
        print(f"  Requires {hops} reasoning hops")
    elif q_type in ['bridge', 'compositional']:
        print(f"  Requires 2 reasoning hops (bridge/compositional)")
    elif q_type == 'comparison':
        print(f"  Requires comparing two entities (2 hops)")

ALL QUESTIONS IN OUR SAMPLE

[1] [HOTPOTQA] [bridge]
  Q: What nationality was Oliver Reed's character in the film Royal Flash?
  A: Prussian
  Answer in context: True
  Requires 2 reasoning hops (bridge/compositional)

[2] [HOTPOTQA] [bridge]
  Q: Pacific Mozart Ensemble performed which German composer's Der Lindberghflug in 2002?
  A: Kurt Julian Weill
  Answer in context: True
  Requires 2 reasoning hops (bridge/compositional)

[3] [HOTPOTQA] [bridge]
  Q: Who released the song "With or Without You" first, Jai McDowall or U2?
  A: U2
  Answer in context: True
  Requires 2 reasoning hops (bridge/compositional)

[4] [MUSIQUE] [3-hop]
  Q: Who is the president of the newly declared independent country that is part of the Commission of Truth and Friendship with the country where Kotamadya is located?
  A: Francisco Guterres
  Answer in context: False
  Requires 3 reasoning hops

[5] [MUSIQUE] [3-hop]
  Q: What is the name of the famous bridge located in the birthplace of the composer of

In [22]:
# Calculate token budget for running experiments with this data
# This tells us how many API tokens step03 will use

def estimate_tokens(text_or_length):
    if isinstance(text_or_length, str):
        return max(1, len(text_or_length) // 4)
    return max(1, int(text_or_length) // 4)

print("=" * 60)
print("TOKEN BUDGET ESTIMATION")
print("=" * 60)

n = len(all_records)   # 9 for n=3 smoke test (3 datasets × 3 questions)

# Per-question token estimates
avg_context = sum(len(r['context']) for r in all_records) / n
avg_question= sum(len(r['question']) for r in all_records) / n

# Prompt structure for answer generation:
# system prompt (~100 tokens) + context + question + answer (~150 tokens)
gen_prompt_tokens = 100 + estimate_tokens(avg_context) + estimate_tokens(avg_question)
gen_output_tokens = 150

# Prompt structure for judge:
# judge prompt (~200 tokens) + context + question + answer = judge output (~100 tokens)
judge_prompt_tokens = 200 + estimate_tokens(avg_context) + gen_output_tokens
judge_output_tokens = 100

# Per question: 4 conditions × (gen + faithfulness judge + relevancy judge)
# + retry possibility (assume 30% trigger rate)
calls_per_question = 4 * (1 + 1 + 1)   # 4 conditions × 3 calls
tokens_per_question = calls_per_question * (
    (gen_prompt_tokens + gen_output_tokens +
     judge_prompt_tokens + judge_output_tokens) / 2
)

total_tokens = int(tokens_per_question * n)
daily_budget = 200_000

print(f"\nSample size          : {n} questions")
print(f"Avg context length   : {avg_context:.0f} chars = ~{estimate_tokens(avg_context)} tokens")
print(f"API calls per Q      : ~{calls_per_question}")
print(f"Tokens per question  : ~{tokens_per_question:.0f}")
print(f"Total tokens (n={n}) : ~{total_tokens:,}")
print(f"Daily TPD budget     : {daily_budget:,}")
print(f"Budget used          : {total_tokens/daily_budget*100:.1f}%")
print()
if total_tokens < daily_budget:
    print("✓ Fits within one day's free tier budget")
else:
    print("⚠  Exceeds daily budget — will need 2+ days")
print()
print("For n=50:  ~", int(tokens_per_question * 50), "tokens (~",
      round(tokens_per_question * 50 / daily_budget * 100), "% of daily budget)")

TOKEN BUDGET ESTIMATION

Sample size          : 9 questions
Avg context length   : 3773 chars = ~943 tokens
API calls per Q      : ~12
Tokens per question  : ~15642
Total tokens (n=9) : ~140,778
Daily TPD budget     : 200,000
Budget used          : 70.4%

✓ Fits within one day's free tier budget

For n=50:  ~ 782100 tokens (~ 391 % of daily budget)


---
## Section 11 — Why These Datasets Prove Our Novelty

### The connection to the paper's claims

Recall the three genuine novelties from the SciGraphAgent paper:

**Novelty 1 — The RAGAS retry gate:**  
We test this on HotpotQA, MuSiQue, and 2WikiMultiHopQA because these benchmarks are specifically designed for questions that require multi-hop reasoning. A standard RAG system (without the retry gate) will fail on many of these because a single retrieval pass misses the second or third hop. The retry gate — which re-retrieves when faithfulness is low — should recover some of these failures.

**Novelty 2 — Hybrid Graph-RAG outperforms vector-only RAG:**  
Han et al. (2025) showed that graph-structured retrieval substantially outperforms vector-only RAG specifically on multi-hop questions. These three benchmarks are the standard way to demonstrate that finding. We need questions that require connecting multiple documents — single-hop questions would not reveal the difference.

**Why the same benchmarks as competing papers:**  
Graph-R1 (ICML 2026) reports F1 of 0.58 on HotpotQA. GraphRAG-R1 (WWW 2026) reports 0.62. By running on the same datasets, our results are directly comparable — even if we cannot claim to beat those numbers.

### What we are NOT claiming

We are not claiming to beat Graph-R1 or GraphRAG-R1. Those systems use reinforcement learning trained on GPU clusters. We are claiming that:
1. The RAGAS retry gate measurably improves faithfulness over a single-pass agent
2. Hybrid retrieval (α=0.6) measurably outperforms vector-only retrieval (α=0.0)
3. Our system achieves this while being free, GPU-free, and pip-installable

That combination — not the absolute F1 score — is the contribution.

In [24]:
# Show published F1 scores on our benchmarks for context
# These are from primary papers cited in Section 5.2 of the article

literature = [
    {"system": "SciGraphAgent (ours)",      "hotpotqa": "TBD", "musique": "TBD", "wiki2": "TBD", "gpu": "✓", "free": "✓", "gate": "✓"},
    {"system": "Self-RAG (ICLR 2024)",      "hotpotqa": "0.450","musique": "N/A",  "wiki2": "N/A",  "gpu": "✓", "free": "✓", "gate": "~"},
    {"system": "Graph-R1 (ICML 2026)",      "hotpotqa": "~0.58","musique": "~0.48","wiki2": "~0.71","gpu": "✓", "free": "✓", "gate": "✗"},
    {"system": "GraphRAG-R1 (WWW 2026)",    "hotpotqa": "~0.62","musique": "~0.55","wiki2": "~0.75","gpu": "✓", "free": "✓", "gate": "✗"},
    {"system": "LightRAG (EMNLP 2025)",     "hotpotqa": "N/A",  "musique": "N/A",  "wiki2": "N/A",  "gpu": "✗", "free": "✓", "gate": "✗"},
    {"system": "MS GraphRAG (2024)",         "hotpotqa": "N/A",  "musique": "N/A",  "wiki2": "N/A",  "gpu": "✗", "free": "✓", "gate": "✗"},
]

print(f"{'System':<28} {'HotpotQA':>10} {'MuSiQue':>9} {'2Wiki':>7} {'GPU-free':>9} {'Open':>6} {'Gate':>5}")
print("-" * 78)
for row in literature:
    print(
        f"{row['system']:<28} "
        f"{row['hotpotqa']:>10} "
        f"{row['musique']:>9} "
        f"{row['wiki2']:>7} "
        f"{row['gpu']:>9} "
        f"{row['free']:>6} "
        f"{row['gate']:>5}"
    )

print()
print("GPU-free = runs without GPU  |  Open = open source  |  Gate = RAGAS retry gate")
print("N/A = not evaluated on this benchmark  |  ~ = approximate  |  TBD = our results")
print()
print("Note: Graph-R1 and GraphRAG-R1 use RL training on GPU clusters.")
print("Our claim is not higher F1 — it is the unique combination of:")
print("  GPU-free + open source + RAGAS gate + multi-agent + deployable")

System                         HotpotQA   MuSiQue   2Wiki  GPU-free   Open  Gate
------------------------------------------------------------------------------
SciGraphAgent (ours)                TBD       TBD     TBD         ✓      ✓     ✓
Self-RAG (ICLR 2024)              0.450       N/A     N/A         ✓      ✓     ~
Graph-R1 (ICML 2026)              ~0.58     ~0.48   ~0.71         ✓      ✓     ✗
GraphRAG-R1 (WWW 2026)            ~0.62     ~0.55   ~0.75         ✓      ✓     ✗
LightRAG (EMNLP 2025)               N/A       N/A     N/A         ✗      ✓     ✗
MS GraphRAG (2024)                  N/A       N/A     N/A         ✗      ✓     ✗

GPU-free = runs without GPU  |  Open = open source  |  Gate = RAGAS retry gate
N/A = not evaluated on this benchmark  |  ~ = approximate  |  TBD = our results

Note: Graph-R1 and GraphRAG-R1 use RL training on GPU clusters.
Our claim is not higher F1 — it is the unique combination of:
  GPU-free + open source + RAGAS gate + multi-agent + deployable


---
## Section 12 — Commit to GitHub

Before committing, we verify the output files are correct and check the git status.

In [25]:
import json, subprocess
from pathlib import Path

print("=" * 55)
print("PRE-COMMIT VERIFICATION")
print("=" * 55)

# 1. Check step01 script exists
script = Path("step01_load_benchmarks.py")
status = "✓" if script.exists() else "✗"
print(f"\n{status} step01_load_benchmarks.py exists")

# 2. Check all three data files were saved
print("\nData files:")
for dataset in ["hotpotqa", "musique", "2wikimultihopqa"]:
    for n in [3, 50]:
        path = Path("data") / f"{dataset}_sample_{n}.json"
        if path.exists():
            with open(path) as f:
                records = json.load(f)
            size_kb = path.stat().st_size // 1024
            print(f"  ✓ {path.name:<40} {len(records):>3} records  {size_kb:>4} KB")

# 3. Validate JSON structure
print("\nStructure validation:")
required_keys = {"id", "question", "answer", "context", "dataset", "q_type"}
all_valid = True
for dataset in ["hotpotqa", "musique", "2wikimultihopqa"]:
    path = Path("data") / f"{dataset}_sample_3.json"
    if path.exists():
        with open(path) as f:
            records = json.load(f)
        for rec in records:
            missing = required_keys - set(rec.keys())
            if missing:
                print(f"  ✗ {dataset}: missing keys {missing}")
                all_valid = False
                break
        else:
            print(f"  ✓ {dataset}: all required keys present")

# 4. Git status
print("\nGit status:")
result = subprocess.run("git status --short", shell=True,
                        capture_output=True, text=True)
for line in result.stdout.strip().split("\n"):
    if line.strip():
        print(" ", line)

print()
if all_valid:
    print("✓ All checks passed — ready to commit")
else:
    print("✗ Fix issues above before committing")

print()
print("Run in terminal:")
print('  git add step01_load_benchmarks.py notebooks/01_load_benchmarks.ipynb')
print('  git commit -m "Add step01 and Notebook 01: benchmark dataset loader')
print("    - HotpotQA: hotpotqa/hotpot_qa (fixed from hotpot_qa)")
print("    - MuSiQue: bdsaglam/musique (fixed from musique)")
print("    - 2WikiMultiHopQA: framolfese/2WikiMultihopQA (fixed from voidful/2wikimqa)")
print('    - Smoke test: 3/3 datasets pass, n=3"')
print("  git push")

PRE-COMMIT VERIFICATION

✓ step01_load_benchmarks.py exists

Data files:
  ✓ hotpotqa_sample_3.json                     3 records    19 KB
  ✓ musique_sample_3.json                      3 records     6 KB
  ✓ 2wikimultihopqa_sample_3.json              3 records     9 KB

Structure validation:
  ✓ hotpotqa: all required keys present
  ✓ musique: all required keys present
  ✓ 2wikimultihopqa: all required keys present

Git status:
  M notebooks/00_setup_and_foundations.ipynb
  ?? 01_load_benchmarks.ipynb
  ?? step01_load_benchmarks.py

✓ All checks passed — ready to commit

Run in terminal:
  git add step01_load_benchmarks.py notebooks/01_load_benchmarks.ipynb
  git commit -m "Add step01 and Notebook 01: benchmark dataset loader
    - HotpotQA: hotpotqa/hotpot_qa (fixed from hotpot_qa)
    - MuSiQue: bdsaglam/musique (fixed from musique)
    - 2WikiMultiHopQA: framolfese/2WikiMultihopQA (fixed from voidful/2wikimqa)
    - Smoke test: 3/3 datasets pass, n=3"
  git push


---
## Summary — What We Learned

### Concepts covered

| Concept | What it is | Where it appears |
|---|---|---|
| Benchmark dataset | Standardised Q&A collection for measuring AI quality | HotpotQA, MuSiQue, 2WikiMultiHop |
| Multi-hop question | Requires connecting facts across 2+ documents | All three datasets |
| F1 score | Token-overlap measure of answer quality (0.0–1.0) | step04_compute_metrics.py |
| HuggingFace | Website hosting open AI models and datasets | load_dataset() |
| datasets library | Python tool for downloading HF datasets | step01_load_benchmarks.py |
| Parquet | Compressed columnar file format for tabular data | HF dataset storage |
| Split (train/val/test) | Standard data partitions in ML | split="validation" |
| Shuffle + seed | Reproducible random sampling | seed=42 throughout |
| JSON | Text format for storing structured data | All .json files in data/ |
| Manifest | Index file pointing to dataset locations | data/manifest.json |
| argparse | Reads command-line arguments (--n 3) | All step scripts |
| `__name__ == "__main__"` | Runs main() only when executed directly | All step scripts |

### Files created by this step

```
data/
├── hotpotqa_sample_3.json       ← 3 HotpotQA questions
├── musique_sample_3.json        ← 3 MuSiQue questions  
├── 2wikimultihopqa_sample_3.json ← 3 2Wiki questions
└── manifest.json                ← index of all saved files
```

### What comes next

**Notebook 02 — Building Retrieval Systems**

We take the downloaded context passages and build two retrieval structures:
1. A **ChromaDB vector index** — converts text to number arrays for similarity search
2. A **knowledge graph** — extracts entities and builds a graph of their relationships

These are the two retrieval signals that the hybrid Graph-RAG system will combine.

---
*Notebook 01 complete. Commit and push, then continue with `02_build_retrieval_systems.ipynb`.*